# DP Auditing Tightness Study — Full Pipeline

**EECE 608 — Mohamad Faour**

This notebook runs the complete experimental pipeline:
1. Upload your repo as a Kaggle dataset
2. Train DP-SGD models across epsilon sweep
3. Run canary + passive audits
4. Produce unified comparison + gap decomposition

**Runtime:** Set to **GPU T4 x2** or **GPU P100** in Kaggle settings.

## 0. Setup

In [ ]:
# Install dependencies (Kaggle has PyTorch pre-installed)
!pip install opacus>=1.5 dp-accounting>=0.4 --quiet

import sys, os

# === OPTION A: If you uploaded the repo as a Kaggle dataset ===
# Uncomment and edit the path:
# REPO_ROOT = '/kaggle/input/eece608-dp-audit/EECE_608'

# === OPTION B: If you cloned from GitHub ===
# !git clone https://github.com/YOUR_USER/EECE_608.git
# REPO_ROOT = '/kaggle/working/EECE_608'

# === OPTION C: If you uploaded as a zip ===
# !unzip /kaggle/input/eece608/EECE_608.zip -d /kaggle/working/
# REPO_ROOT = '/kaggle/working/EECE_608'

# For now, set this to wherever your repo lives:
REPO_ROOT = '/kaggle/working/EECE_608'

sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))
sys.path.insert(0, os.path.join(REPO_ROOT, 'experiments'))
os.chdir(REPO_ROOT)

print(f'Working directory: {os.getcwd()}')
print(f'GPU available: {__import__("torch").cuda.is_available()}')
print(f'Device: {__import__("torch").cuda.get_device_name(0) if __import__("torch").cuda.is_available() else "CPU"}')

In [ ]:
# Verify imports
from dp_audit_tightness.config import load_train_config
from dp_audit_tightness.data.datasets import load_dataset_bundle
from dp_audit_tightness.models.simple_mlp import build_model
from dp_audit_tightness.training.dp_sgd import train_dp_sgd
from dp_audit_tightness.evaluation.gap_decomposition import decompose_gap
from dp_audit_tightness.privacy.pld_accounting import compute_epsilon_pld
print('All imports OK (including PLD accounting)')

## 1. Smoke Test — Verify Pipeline on MNIST

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger('pipeline')

# Quick MNIST smoke test
config = load_train_config('configs/train/mnist_mlp_dp_sgd_smoke.yaml')
outcome = train_dp_sgd(config, logger)

print(f'\n=== MNIST Smoke Test ===')
print(f'Accuracy: {outcome.record.utility_metrics["accuracy"]:.4f}')
print(f'epsilon_upper (RDP): {outcome.record.epsilon_upper_theory:.4f}')
print(f'epsilon_upper (PLD): {outcome.record.epsilon_upper_pld:.4f}' if outcome.record.epsilon_upper_pld else 'epsilon_upper (PLD): N/A')
print(f'PLD backend: {outcome.record.pld_accounting_backend}')
if outcome.record.epsilon_upper_pld:
    gap = outcome.record.epsilon_upper_theory - outcome.record.epsilon_upper_pld
    print(f'Accounting gap (RDP - PLD): {gap:.4f} ({gap/outcome.record.epsilon_upper_theory*100:.1f}% of RDP bound)')
print(f'Checkpoint: {outcome.checkpoint_path}')

## 2. Smoke Test — Verify CIFAR-10 Pipeline

In [ ]:
config_cifar = load_train_config('configs/train/cifar10_cnn_dp_sgd_smoke.yaml')
outcome_cifar = train_dp_sgd(config_cifar, logger)

print(f'\n=== CIFAR-10 Smoke Test ===')
print(f'Accuracy: {outcome_cifar.record.utility_metrics["accuracy"]:.4f}')
print(f'epsilon_upper (RDP): {outcome_cifar.record.epsilon_upper_theory:.4f}')
print(f'epsilon_upper (PLD): {outcome_cifar.record.epsilon_upper_pld:.4f}' if outcome_cifar.record.epsilon_upper_pld else 'epsilon_upper (PLD): N/A')
print(f'PLD backend: {outcome_cifar.record.pld_accounting_backend}')
if outcome_cifar.record.epsilon_upper_pld:
    gap = outcome_cifar.record.epsilon_upper_theory - outcome_cifar.record.epsilon_upper_pld
    print(f'Accounting gap (RDP - PLD): {gap:.4f} ({gap/outcome_cifar.record.epsilon_upper_theory*100:.1f}% of RDP bound)')
print(f'Checkpoint: {outcome_cifar.checkpoint_path}')

## 3. Epsilon Sweep — Generate Configs

In [ ]:
from generate_sweep_configs import generate_sweep_configs

# Noise multipliers that give a range of theoretical epsilon values.
# Lower sigma = higher epsilon (less privacy, tighter audits expected).
# Higher sigma = lower epsilon (more privacy, larger gap expected).
NOISE_MULTIPLIERS = [0.5, 0.8, 1.1, 1.5, 2.0]
SEEDS = [123, 124, 125]

# MNIST sweep
mnist_sweep = generate_sweep_configs(
    'configs/train/mnist_mlp_dp_sgd_smoke.yaml',
    noise_multipliers=NOISE_MULTIPLIERS,
    training_seeds=SEEDS,
    output_dir='configs/sweep/mnist',
    epochs_override=5,
)
print(f'Generated {len(mnist_sweep)} MNIST configs')

# CIFAR-10 sweep
cifar_sweep = generate_sweep_configs(
    'configs/train/cifar10_cnn_dp_sgd_smoke.yaml',
    noise_multipliers=NOISE_MULTIPLIERS,
    training_seeds=SEEDS,
    output_dir='configs/sweep/cifar10',
    epochs_override=5,
)
print(f'Generated {len(cifar_sweep)} CIFAR-10 configs')

## 4. Run Epsilon Sweep — Training

In [ ]:
import json
from pathlib import Path
from dp_audit_tightness.utils.results import save_training_result

def run_sweep_training(config_paths, label=''):
    """Train one model per config, save results, return records."""
    records = []
    for i, cfg_path in enumerate(config_paths):
        cfg = load_train_config(cfg_path)
        print(f'\n[{label} {i+1}/{len(config_paths)}] '
              f'sigma={cfg.training.noise_multiplier}, seed={cfg.run.training_seed}')
        try:
            outcome = train_dp_sgd(cfg, logger)
            save_training_result(outcome.record, results_root=cfg.run.results_root)
            records.append(outcome.record)
            pld_str = f', eps_pld={outcome.record.epsilon_upper_pld:.4f}' if outcome.record.epsilon_upper_pld else ''
            print(f'  acc={outcome.record.utility_metrics["accuracy"]:.4f}, '
                  f'eps_rdp={outcome.record.epsilon_upper_theory:.4f}{pld_str}')
        except Exception as e:
            print(f'  FAILED: {e}')
    return records

print('=== MNIST Epsilon Sweep ===')
mnist_records = run_sweep_training(mnist_sweep, label='MNIST')

In [ ]:
print('=== CIFAR-10 Epsilon Sweep ===')
cifar_records = run_sweep_training(cifar_sweep, label='CIFAR-10')

## 5. Run Audits on Each Trained Model

In [ ]:
from dp_audit_tightness.config import load_train_config_snapshot
from dp_audit_tightness.auditing.passive.auditors import run_passive_audit_repeated
from dp_audit_tightness.privacy.empirical import estimate_empirical_lower_bound
from dp_audit_tightness.evaluation.metrics import compute_privacy_loss_gap, compute_tightness_ratio
from dp_audit_tightness.utils.results import AuditRunRecord, save_audit_result, PASSIVE_AUDIT_REGIME
from dp_audit_tightness.utils.paths import build_run_id

PASSIVE_SEEDS = [401, 402, 403]
QUERY_BUDGET = 1024

def run_passive_audits_for_records(training_records, dataset_name):
    """Run passive audits for a list of training records."""
    audit_results = []
    for i, rec in enumerate(training_records):
        print(f'\n[Passive {dataset_name} {i+1}/{len(training_records)}] '
              f'sigma={rec.noise_multiplier}, seed={rec.training_seed}')
        
        # Load the training config snapshot
        config_path = Path(rec.model_artifact_path).parent.parent / 'configs' / f'{rec.training_run_id}_config.json'
        if not config_path.exists():
            print(f'  Skipping (no config snapshot at {config_path})')
            continue
        
        try:
            training_config = load_train_config_snapshot(config_path)
            observations = run_passive_audit_repeated(
                training_record=rec,
                training_config=training_config,
                score_type='logit_margin',
                calibration_method='none',
                query_budget=QUERY_BUDGET,
                repeated_seeds=PASSIVE_SEEDS,
            )
            best_obs = max(observations, key=lambda o: len(o.member_scores))
            estimate = estimate_empirical_lower_bound(
                member_scores=best_obs.member_scores,
                nonmember_scores=best_obs.nonmember_scores,
                delta=rec.delta,
                align_event_to_score_direction=True,
                require_member_favoring=True,
            )
            eps_lower = estimate.epsilon_lower_empirical
            gap = compute_privacy_loss_gap(rec.epsilon_upper_theory, eps_lower)
            ratio = compute_tightness_ratio(eps_lower, rec.epsilon_upper_theory)
            print(f'  eps_lower={eps_lower:.4f}, ratio={ratio:.4f}')
            audit_results.append({
                'training_run_id': rec.training_run_id,
                'noise_multiplier': rec.noise_multiplier,
                'epsilon_upper': rec.epsilon_upper_theory,
                'epsilon_lower_passive': eps_lower,
                'tightness_ratio_passive': ratio,
                'gap_passive': gap,
            })
        except Exception as e:
            print(f'  FAILED: {e}')
    return audit_results

print('=== Passive Audits: MNIST ===')
mnist_passive = run_passive_audits_for_records(mnist_records, 'MNIST')

In [ ]:
print('=== Passive Audits: CIFAR-10 ===')
cifar_passive = run_passive_audits_for_records(cifar_records, 'CIFAR-10')

## 6. Collect Results & Gap Decomposition

In [ ]:
import pandas as pd
import numpy as np

def build_results_table(training_records, passive_results, dataset_name):
    """Aggregate training + audit results into a DataFrame with full gap decomposition."""
    rows = []
    passive_lookup = {r['training_run_id']: r for r in passive_results}
    
    for rec in training_records:
        eps_rdp = rec.epsilon_upper_theory
        eps_pld = rec.epsilon_upper_pld
        
        row = {
            'dataset': dataset_name,
            'model': rec.model_name,
            'noise_multiplier': rec.noise_multiplier,
            'epochs': rec.epochs,
            'seed': rec.training_seed,
            'epsilon_upper_rdp': eps_rdp,
            'epsilon_upper_pld': eps_pld,
            'pld_backend': rec.pld_accounting_backend,
            'accuracy': rec.utility_metrics.get('accuracy', None),
            'loss': rec.utility_metrics.get('loss', None),
        }
        
        # Accounting gap (RDP vs PLD)
        if eps_pld is not None:
            row['accounting_gap'] = round(eps_rdp - eps_pld, 6)
            row['accounting_gap_pct'] = round((eps_rdp - eps_pld) / eps_rdp * 100, 2) if eps_rdp > 0 else None
        else:
            row['accounting_gap'] = None
            row['accounting_gap_pct'] = None
        
        # Tightest upper bound
        tightest_upper = eps_pld if eps_pld is not None else eps_rdp
        
        # Add passive results if available
        passive = passive_lookup.get(rec.training_run_id, {})
        eps_lower = passive.get('epsilon_lower_passive', None)
        row['epsilon_lower_passive'] = eps_lower
        
        if eps_lower is not None:
            row['tightness_ratio_vs_rdp'] = round(eps_lower / eps_rdp, 6) if eps_rdp > 0 else None
            row['tightness_ratio_vs_pld'] = round(eps_lower / tightest_upper, 6) if tightest_upper > 0 else None
            row['total_gap'] = round(eps_rdp - eps_lower, 6)
            row['residual_gap'] = round(tightest_upper - eps_lower, 6)
        else:
            row['tightness_ratio_vs_rdp'] = None
            row['tightness_ratio_vs_pld'] = None
            row['total_gap'] = None
            row['residual_gap'] = None
        
        rows.append(row)
    
    return pd.DataFrame(rows)

df_mnist = build_results_table(mnist_records, mnist_passive, 'MNIST')
df_cifar = build_results_table(cifar_records, cifar_passive, 'CIFAR-10')
df_all = pd.concat([df_mnist, df_cifar], ignore_index=True)

# Aggregate across seeds
agg_cols = {
    'epsilon_upper_rdp': 'mean',
    'epsilon_upper_pld': 'mean',
    'accounting_gap': 'mean',
    'accounting_gap_pct': 'mean',
    'accuracy': ['mean', 'std'],
    'epsilon_lower_passive': ['mean', 'std'],
    'tightness_ratio_vs_rdp': ['mean', 'std'],
    'tightness_ratio_vs_pld': ['mean', 'std'],
    'total_gap': 'mean',
    'residual_gap': 'mean',
}
df_agg = df_all.groupby(['dataset', 'model', 'noise_multiplier', 'epochs']).agg(agg_cols).round(4).reset_index()

print(df_agg.to_string())

# Save
df_all.to_csv('results/sweep_all_results.csv', index=False)
df_agg.to_csv('results/sweep_aggregated.csv', index=False)
print(f'\nSaved results to results/sweep_all_results.csv and results/sweep_aggregated.csv')

## 7. Key Figures

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12, 'figure.figsize': (10, 6)})

# --- Figure 1: Tightness Ratio vs Theoretical Epsilon ---
# Shows ratio against both RDP and PLD upper bounds
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

for dataset_name, marker, color in [('MNIST', 'o', '#2196F3'), ('CIFAR-10', 's', '#FF5722')]:
    subset = df_all[df_all['dataset'] == dataset_name]
    if subset.empty:
        continue
    
    # Ratio vs RDP (solid)
    grouped_rdp = subset.groupby('noise_multiplier').agg({
        'epsilon_upper_rdp': 'mean',
        'tightness_ratio_vs_rdp': ['mean', 'std'],
    }).dropna()
    if not grouped_rdp.empty:
        x = grouped_rdp[('epsilon_upper_rdp', 'mean')].values
        y = grouped_rdp[('tightness_ratio_vs_rdp', 'mean')].values
        yerr = grouped_rdp[('tightness_ratio_vs_rdp', 'std')].values
        ax.errorbar(x, y, yerr=yerr, marker=marker, color=color,
                    label=f'{dataset_name} vs RDP', linewidth=2,
                    markersize=8, capsize=4)
    
    # Ratio vs PLD (dashed — shows the gap is smaller when accounting is tighter)
    grouped_pld = subset.groupby('noise_multiplier').agg({
        'epsilon_upper_rdp': 'mean',
        'tightness_ratio_vs_pld': ['mean', 'std'],
    }).dropna()
    if not grouped_pld.empty:
        x2 = grouped_pld[('epsilon_upper_rdp', 'mean')].values
        y2 = grouped_pld[('tightness_ratio_vs_pld', 'mean')].values
        yerr2 = grouped_pld[('tightness_ratio_vs_pld', 'std')].values
        ax.errorbar(x2, y2, yerr=yerr2, marker=marker, color=color,
                    label=f'{dataset_name} vs PLD', linewidth=2,
                    markersize=8, capsize=4, linestyle='--', alpha=0.7)

ax.set_xlabel(r'$\varepsilon_{upper}$ (theoretical, RDP)', fontsize=14)
ax.set_ylabel(r'Tightness ratio ($\varepsilon_{lower} / \varepsilon_{upper}$)', fontsize=14)
ax.set_title('Tightness Ratio vs. Privacy Budget (RDP vs PLD Upper Bound)', fontsize=15)
ax.legend(fontsize=11)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/figure_tightness_vs_epsilon.pdf', dpi=300, bbox_inches='tight')
plt.savefig('results/figure_tightness_vs_epsilon.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved figure_tightness_vs_epsilon.pdf/.png')

In [ ]:
# --- Figure 2: Privacy-Utility Tradeoff ---
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

for dataset_name, marker, color in [('MNIST', 'o', '#2196F3'), ('CIFAR-10', 's', '#FF5722')]:
    subset = df_all[df_all['dataset'] == dataset_name]
    if subset.empty:
        continue
    grouped = subset.groupby('noise_multiplier').agg({
        'epsilon_upper_rdp': 'mean',
        'accuracy': ['mean', 'std'],
    })
    
    x = grouped[('epsilon_upper_rdp', 'mean')].values
    y = grouped[('accuracy', 'mean')].values * 100
    yerr = grouped[('accuracy', 'std')].values * 100
    
    ax.errorbar(x, y, yerr=yerr, marker=marker, color=color,
                label=dataset_name, linewidth=2, markersize=8, capsize=4)

ax.set_xlabel(r'$\varepsilon_{upper}$ (theoretical, RDP)', fontsize=14)
ax.set_ylabel('Test Accuracy (%)', fontsize=14)
ax.set_title('Privacy-Utility Tradeoff', fontsize=16)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/figure_privacy_utility.pdf', dpi=300, bbox_inches='tight')
plt.savefig('results/figure_privacy_utility.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved figure_privacy_utility.pdf/.png')

In [ ]:
# --- Figure 3: Full Gap Decomposition (3-component stacked bar) ---
# total_gap = accounting_gap + residual_gap + recovered (empirical lower bound)
# The stacked bar shows: [recovered | residual | accounting_gap]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, dataset_name in zip(axes, ['MNIST', 'CIFAR-10']):
    subset = df_all[df_all['dataset'] == dataset_name].dropna(subset=['epsilon_lower_passive'])
    if subset.empty:
        ax.set_title(f'{dataset_name} (no audit data)')
        continue
    
    grouped = subset.groupby('noise_multiplier').agg({
        'epsilon_upper_rdp': 'mean',
        'epsilon_upper_pld': 'mean',
        'epsilon_lower_passive': 'mean',
        'accounting_gap': 'mean',
        'residual_gap': 'mean',
    }).sort_values('epsilon_upper_rdp')
    
    x_labels = [f'σ={sigma}\nε_rdp={eps:.2f}'
                for sigma, eps in zip(grouped.index, grouped['epsilon_upper_rdp'])]
    x = range(len(x_labels))
    
    recovered = grouped['epsilon_lower_passive'].values
    residual = grouped['residual_gap'].values
    accounting = grouped['accounting_gap'].values
    
    # Stack: recovered (bottom) + residual (middle) + accounting (top)
    ax.bar(x, recovered, label='Recovered (ε_lower)', color='#4CAF50', alpha=0.85)
    ax.bar(x, residual, bottom=recovered,
           label='Residual gap (worst-case, attack limit)', color='#FF9800', alpha=0.75)
    ax.bar(x, accounting, bottom=recovered + residual,
           label='Accounting gap (RDP → PLD)', color='#F44336', alpha=0.65)
    
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, fontsize=10)
    ax.set_title(dataset_name, fontsize=14)
    ax.set_ylabel(r'$\varepsilon$', fontsize=14)
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('Gap Decomposition: Where Does the Tightness Gap Come From?', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('results/figure_gap_decomposition.pdf', dpi=300, bbox_inches='tight')
plt.savefig('results/figure_gap_decomposition.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved figure_gap_decomposition.pdf/.png')

In [ ]:
# --- Figure 4: Accounting Gap as % of Total Gap ---
# Key finding: what fraction of the tightness gap is purely due to loose accounting?

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

for dataset_name, marker, color in [('MNIST', 'o', '#2196F3'), ('CIFAR-10', 's', '#FF5722')]:
    subset = df_all[(df_all['dataset'] == dataset_name)].dropna(subset=['accounting_gap', 'total_gap'])
    if subset.empty:
        continue
    grouped = subset.groupby('noise_multiplier').agg({
        'epsilon_upper_rdp': 'mean',
        'accounting_gap': 'mean',
        'total_gap': 'mean',
        'residual_gap': 'mean',
    })
    
    x = grouped['epsilon_upper_rdp'].values
    acct_pct = (grouped['accounting_gap'] / grouped['total_gap'] * 100).values
    resid_pct = (grouped['residual_gap'] / grouped['total_gap'] * 100).values
    
    ax.plot(x, acct_pct, marker=marker, color=color, linewidth=2, markersize=8,
            label=f'{dataset_name} — accounting gap %')
    ax.plot(x, resid_pct, marker=marker, color=color, linewidth=2, markersize=8,
            linestyle='--', alpha=0.6, label=f'{dataset_name} — residual gap %')

ax.set_xlabel(r'$\varepsilon_{upper}$ (RDP)', fontsize=14)
ax.set_ylabel('Share of Total Gap (%)', fontsize=14)
ax.set_title('What Fraction of the Tightness Gap Is Due to Loose Accounting?', fontsize=15)
ax.legend(fontsize=11)
ax.set_ylim(0, 105)
ax.axhline(y=50, color='gray', linestyle=':', alpha=0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/figure_accounting_gap_share.pdf', dpi=300, bbox_inches='tight')
plt.savefig('results/figure_accounting_gap_share.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved figure_accounting_gap_share.pdf/.png')

## 8. Download Results

All outputs are saved under `results/`. Download them from the Kaggle output panel:
- `sweep_all_results.csv` — raw per-run data (with RDP + PLD epsilon, gap decomposition)
- `sweep_aggregated.csv` — seed-averaged summary
- `figure_tightness_vs_epsilon.pdf` — tightness ratio vs epsilon (RDP and PLD)
- `figure_privacy_utility.pdf` — accuracy vs epsilon tradeoff
- `figure_gap_decomposition.pdf` — 3-component stacked gap chart (accounting + residual + recovered)
- `figure_accounting_gap_share.pdf` — what % of the gap is loose accounting vs. real privacy leakage

In [ ]:
# List all output files
from pathlib import Path
for f in sorted(Path('results').rglob('*')):
    if f.is_file():
        size = f.stat().st_size / 1024
        print(f'  {f} ({size:.1f} KB)')